In [ ]:
import io
import os
import wave
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from google.genai import types
from PIL import Image
from pydantic import BaseModel, Field

load_dotenv()
GEMINI_KEY = os.getenv("API_KEY")
client = genai.Client(api_key=GEMINI_KEY)

fp = "soul.md"
with open(fp, "r") as file:
    soul = file.read()

SYSTEM_PROMPT = f"""
You are creating a childrens story 
{soul}
"""


class Story(BaseModel):
    title: str = Field(description="The title of the story")
    summary: str = Field(description="An overview of the story for the back cover")
    pages: list[str] = Field(
        description="A list of 4 strings. Each string is a page of the story."
    )


def generate_story(contents: list, model: str = "gemini-3.1-pro-preview") -> Story:
    story_response = client.models.generate_content(
        model=model,
        contents=contents,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=Story,
            system_instruction=SYSTEM_PROMPT,
        ),
    )
    return story_response.parsed


def generate_image(contents: list, model: str = "gemini-3-pro-image-preview") -> Image:
    image_response = client.models.generate_content(
        model=model,
        contents=contents,
        config=types.GenerateContentConfig(
            response_modalities=["IMAGE"],
            system_instruction=SYSTEM_PROMPT,
        ),
    )
    # print(image_response)
    img_data = image_response.candidates[0].content.parts[0].inline_data.data
    img = Image.open(io.BytesIO(img_data))
    return img


def generate_illustration(page: str, characters: Image) -> Image:
    return generate_image(
        [
            f"""
    now draw an illustration for this page of the book.

    page text: {page}

    draw the text on the illustration exactly. make sure to not hallucinate and use the exact copy. 

    also including the characters in the prompt as an image. 

    """,
            characters,
        ]
    )


